# 03 · Snow reliability for Lorenzo's pilot resorts

**Owner** Matthias · **Source roster** Lorenzo's GitHub commit `950cc799c2467e86a76d9d85225367641a8ad41d`

Open this notebook from `notebooks/` in the group project and click **Run All**. It reads `data/processed/pilot_resorts_10.csv` and enriches it with `data/processed/pilot_resort_locations.csv`. The notebook runs the base/top pilot and then the 20 study winters for all ten locations. Saved API responses let a repeat run finish from the cache.

The study counts the days from **1 December through 15 April** whose highest hourly snow depth reaches **0.30 m**. Winters are labelled by the year they end. The two periods are 1985/86–1994/95 and 2015/16–2024/25.

**Input decisions** · Resort IDs come from the source URLs in Lorenzo's scraper. The coordinates are mapped reference points in the base towns, documented in the locations CSV. The CSV's numeric resort properties are carried through from Lorenzo's output. His code uses preset fallback values, and those fields need review before the final analysis. See `docs/matthias_snow_run.md`.

**Course connection** · Week 1, PDF pp. 49–54 covers API collection. Week 2, p. 23 covers DataFrames and pp. 44–45 cover CSV and JSON.

## 1. Load the shared roster

The notebook first reads Lorenzo's `data/processed/pilot_resorts_10.csv`. It can also accept a manually supplied `data/raw/resorts_pilot_input.csv` when the shared file is absent. It accepts his column names `resort_name`, `elevation_base_m` and `elevation_top_m`.

The companion `pilot_resort_locations.csv` matches rows by resort name and country. It supplies the URL-based resort ID and the coordinates. The IDs connect the snow outputs to the input resorts. If a future roster has missing coordinates, the notebook creates `data/raw/resorts_pilot_coordinates.csv` with map links for review.

“Chamonix” is located at Les Houches because that is the source URL in Lorenzo's notebook. Les 2 Alpes is represented by the lower Mont-de-Lans area. Zermatt and Cervinia remain separate pilot locations on the two sides of the same cross-border ski system.

The project used one-time, cached Nominatim searches, with one machine, a project User-Agent and at least 1.2 seconds between requests. Candidate results were checked by place name and country. The reference points represent base towns. [Nominatim usage policy](https://operations.osmfoundation.org/policies/nominatim/) · © OpenStreetMap contributors, ODbL.

In [2]:
import hashlib
import json
from pathlib import Path
from time import sleep
from urllib.error import HTTPError
from urllib.parse import urlencode
from urllib.request import urlopen

import pandas as pd
from IPython.display import HTML, display

PILOT_WINTERS = (1995, 2025)
STUDY_WINTERS = (*range(1986, 1996), *range(2016, 2026))
country_names = {"CH": "Switzerland", "IT": "Italy", "AT": "Austria", "FR": "France"}

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
raw_dir = root / "data" / "raw"
processed_dir = root / "data" / "processed"
cache_dir = raw_dir / "open_meteo"
cache_dir.mkdir(parents=True, exist_ok=True)
team_candidates = [processed_dir / "pilot_resorts_10.csv", raw_dir / "resorts_pilot_input.csv"]
team_csv = next((path for path in team_candidates if path.is_file()), team_candidates[0])
locations_csv = processed_dir / "pilot_resort_locations.csv"
coordinate_csv = raw_dir / "resorts_pilot_coordinates.csv"

coordinate_columns = ["lat", "lon"]
aliases = {
    "resort_name": "name",
    "country_code": "country",
    "base_elevation_m": "elev_base_m",
    "elevation_base_m": "elev_base_m",
    "top_elevation_m": "elev_top_m",
    "elevation_top_m": "elev_top_m",
    "latitude": "lat",
    "longitude": "lon",
}


def read_text_csv(path):
    table = pd.read_csv(path, sep=None, engine="python", dtype="string", encoding="utf-8-sig", keep_default_na=False)
    table.columns = table.columns.str.strip().str.lower().str.replace(r"\s+", "_", regex=True)
    table = table.rename(columns=aliases)
    if table.columns.duplicated().any():
        raise ValueError(f"{path.name} has duplicate column names.")
    return table


def numeric_column(values):
    # Accept decimal dots and Excel-style decimal commas.
    return pd.to_numeric(values.str.strip().str.replace(",", ".", regex=False), errors="coerce")


def clean_coordinates(table):
    for column, limits in {"lat": (40, 50), "lon": (0, 20)}.items():
        values = numeric_column(table[column].astype("string"))
        table[column] = values.where(values.between(*limits))


def read_resorts(path, require_top=True):
    elevations = ["elev_base_m", "elev_top_m"] if require_top else ["elev_base_m"]
    required = ["resort_id", "name", "country", *elevations]
    table = read_text_csv(path)
    if locations_csv.is_file() and {"name", "country"}.issubset(table):
        locations = read_text_csv(locations_csv)
        join_columns = ["name", "country"]
        supplied_columns = ["resort_id", "lat", "lon"]
        table = table.merge(
            locations[[*join_columns, *supplied_columns]],
            on=join_columns, how="left", validate="one_to_one", suffixes=("", "_location"),
        )
        for column in supplied_columns:
            extra = f"{column}_location"
            if extra in table:
                table[column] = table[column].replace("", pd.NA).fillna(table.pop(extra))
    absent = [column for column in required if column not in table]
    if absent:
        raise ValueError(f"{path.name} needs these columns: {', '.join(absent)}.")
    table = table.reindex(columns=[*required, *coordinate_columns])
    for column in ("resort_id", "name", "country"):
        table[column] = table[column].str.strip()
    if table.empty or table[required[:3]].fillna("").eq("").any().any():
        raise ValueError(f"Fill in the resort ID, name and country for every row in {path.name}.")
    if table["resort_id"].duplicated().any():
        raise ValueError(f"Use each resort ID only once in {path.name}.")

    country_codes = {name.upper(): code for code, name in country_names.items()}
    table["country"] = table["country"].str.upper().replace(country_codes)
    if not table["country"].isin(country_names).all():
        raise ValueError(f"Use CH, IT, AT or FR in the country column of {path.name}.")
    for column in elevations:
        values = numeric_column(table[column])
        if values.isna().any() or values.mod(1).ne(0).any():
            raise ValueError(f"{column} in {path.name} must contain whole numbers in meters.")
        table[column] = values.astype(int)
    if require_top and table["elev_top_m"].lt(table["elev_base_m"]).any():
        raise ValueError("The top elevation must be at or above the base elevation.")
    clean_coordinates(table)
    return table


team_mode = team_csv.is_file()
if team_mode:
    resorts = read_resorts(team_csv)
    if len(resorts) != 10:
        raise ValueError(f"The pilot needs Lorenzo's 10 resorts. This CSV has {len(resorts)}.")

    if resorts[coordinate_columns].isna().any().any() and coordinate_csv.is_file():
        coordinates = read_text_csv(coordinate_csv)
        if not {"resort_id", *coordinate_columns}.issubset(coordinates):
            raise ValueError("Keep the resort_id, lat and lon columns in the coordinate checklist.")
        coordinates["resort_id"] = coordinates["resort_id"].str.strip()
        if coordinates["resort_id"].duplicated().any():
            raise ValueError("Use each resort ID only once in the coordinate checklist.")
        clean_coordinates(coordinates)
        coordinates = coordinates.set_index("resort_id")
        for column in coordinate_columns:
            resorts[column] = resorts[column].fillna(resorts["resort_id"].map(coordinates[column]))
else:
    resorts = pd.DataFrame([{
        "resort_id": "furkel-trafoi-stilfs",
        "name": "Furkel – Trafoi (Stilfs)",
        "country": "IT",
        "elev_base_m": 1530,
        "elev_top_m": 2400,
        "lat": 46.5527404,
        "lon": 10.5106215,
    }])

missing_coordinates = resorts[coordinate_columns].isna().any(axis=1)
pilot_ready = not missing_coordinates.any()
if pilot_ready:
    mode = "Team pilot" if team_mode else "Trafoi example"
    print(f"{mode}: {len(resorts)} resort(s) for winters {PILOT_WINTERS}.")
    display(resorts)
else:
    checklist = resorts[["resort_id", "name", "country", *coordinate_columns]].copy()
    checklist["map_search"] = [
        "https://www.openstreetmap.org/search?" + urlencode({"query": f"{row.name}, {country_names[row.country]}"})
        for row in checklist.itertuples(index=False)
    ]
    checklist.to_csv(coordinate_csv, index=False)
    print(f"{missing_coordinates.sum()} resort(s) need a base-area coordinate.")
    print("Fill the blanks in data/raw/resorts_pilot_coordinates.csv, then run all cells again.")
    display(HTML(checklist.loc[missing_coordinates].to_html(index=False, render_links=True)))

,resort_id,name,country,elev_base_m,elev_top_m,lat,lon
0,zermatt-matterhorn,Zermatt-Matterhorn,CH,1562,3899,46.021208,7.749254
1,st-moritz-corviglia,St. Moritz - Corviglia,CH,1772,3057,46.497896,9.839243
2,andermatt-sedrun-disentis,Andermatt-Sedrun,CH,1444,2961,46.63405,8.594815
3,breuil-cervinia-valtournenche-zermatt,Cervinia-Valtournenche,IT,1562,3899,45.876979,7.622792
4,livigno,Livigno,IT,1816,2798,46.538173,10.135993
5,aprica,Aprica,IT,1162,2270,46.154014,10.152581
6,kitzbuehel-kirchberg,Kitzbuehel,AT,800,2000,47.446359,12.391147
7,soelden,Soelden,AT,1350,3340,46.966632,11.007285
8,les-houches-saint-gervais-chamonix,Chamonix,FR,1000,1900,45.891508,6.799138
9,les-2-alpes,Les 2 Alpes,FR,1280,3523,45.035296,6.129654


## 2. Ask Open-Meteo for hourly snow depth

An **API** lets Python ask a website for data. Our request specifies the coordinates, elevation, winter dates, and the **ERA5-Land** model. The reply is **JSON**, which Python turns into a table with one snow-depth value per hour.

The notebook uses UTC days consistently. It checks the dates, units and all hourly values before counting. An incomplete season raises an error so missing snow readings cannot lower the count. Raw replies are cached in `data/raw/open_meteo/` so rerunning a completed request reads it from disk.

The base and top requests use the **same coordinates** and change only `elevation`. This isolates the effect of the query parameter. The grid returned by the API can sit kilometres away from the lift. [Open-Meteo Historical Weather API](https://open-meteo.com/en/docs/historical-weather-api)

In [3]:
archive_url = "https://archive-api.open-meteo.com/v1/archive"

def snow_request(resort, winter, elevation_m):
    start = f"{winter - 1}-12-01"
    end = f"{winter}-04-15"
    params = {
        "latitude": float(resort.lat),
        "longitude": float(resort.lon),
        "elevation": int(elevation_m),
        "start_date": start,
        "end_date": end,
        "hourly": "snow_depth",
        "models": "era5_land",
        "timezone": "GMT",
        "cell_selection": "land",
    }
    query = urlencode(params)
    cache_id = hashlib.sha256(query.encode()).hexdigest()[:16]
    cache_file = cache_dir / f"{cache_id}.json"
    return f"{archive_url}?{query}", cache_file, start, end


def get_hourly_snow(resort, winter, elevation_m):
    url, cache_file, start, end = snow_request(resort, winter, elevation_m)
    from_cache = cache_file.is_file()
    if from_cache:
        body = cache_file.read_bytes()
    else:
        for attempt in range(3):
            try:
                with urlopen(url, timeout=45) as response:
                    body = response.read()
                break
            except HTTPError as error:
                if error.code not in (500, 502, 503, 504) or attempt == 2:
                    raise
                sleep(2 * (attempt + 1))

    payload = json.loads(body)
    label = f"{resort.resort_id}, winter {winter}, {elevation_m} m"
    if payload.get("hourly_units", {}).get("snow_depth") != "m":
        raise ValueError(f"{label}: expected hourly snow depth in meters.")
    if payload["utc_offset_seconds"] != 0:
        raise ValueError(f"{label}: expected UTC timestamps.")

    hourly = payload["hourly"]
    times = pd.to_datetime(hourly["time"], utc=True)
    depth = pd.Series(hourly["snow_depth"], index=times, dtype="float64")
    expected_times = pd.date_range(start, f"{end} 23:00", freq="h", tz="UTC")
    if not depth.index.equals(expected_times) or depth.isna().any():
        raise ValueError(f"{label}: hourly observations are missing or irregular.")

    if not from_cache:
        temp_file = cache_file.with_suffix(".tmp")
        temp_file.write_bytes(body)
        temp_file.replace(cache_file)

    grid = (payload["latitude"], payload["longitude"])
    return depth, grid


def count_snow_days(hourly_depth):
    daily_max = hourly_depth.resample("D").max()  # One value per day.
    return int(daily_max.ge(0.30).sum())            # Count days at or above 30 cm.

## 3. Compare base and top

For each resort and winter, look at `same_grid_cell` and `same_hourly_series` first. The two snow-day counts can match even if some hourly values differ. If both checks are true, the two elevation requests gave the same model series for that resort and winter.

Only the **base** rows go into the main `snow_winters` table. The top results are a pilot check. The data dictionary calls the elevation sent to the API `elev_query_m`. The comparison also records the location of the grid cell used by the model.

In [4]:
def base_winter_row(resort, winter, snow_days):
    return {
        "resort_id": resort.resort_id,
        "winter": winter,
        "period": "past" if winter <= 1995 else "recent",
        "snow_days_30cm": snow_days,
        "elev_query_m": int(resort.elev_base_m),
    }


if pilot_ready:
    snow_rows = []
    comparisons = []

    for resort in resorts.itertuples(index=False):
        for winter in PILOT_WINTERS:
            base, base_grid = get_hourly_snow(resort, winter, resort.elev_base_m)
            top, top_grid = get_hourly_snow(resort, winter, resort.elev_top_m)
            base_days = count_snow_days(base)
            top_days = count_snow_days(top)

            snow_rows.append(base_winter_row(resort, winter, base_days))
            comparisons.append({
                "resort_id": resort.resort_id,
                "name": resort.name,
                "country": resort.country,
                "winter": winter,
                "lat": resort.lat,
                "lon": resort.lon,
                "base_elev_m": int(resort.elev_base_m),
                "top_elev_m": int(resort.elev_top_m),
                "base_days_30cm": base_days,
                "top_days_30cm": top_days,
                "top_minus_base_days": top_days - base_days,
                "same_grid_cell": base_grid == top_grid,
                "same_hourly_series": base.equals(top),
                "base_grid_lat": base_grid[0],
                "base_grid_lon": base_grid[1],
                "top_grid_lat": top_grid[0],
                "top_grid_lon": top_grid[1],
            })
            print(f"{resort.name} · {winter}: {base_days} base days, {top_days} top days")

    snow_winters = pd.DataFrame(snow_rows)
    comparison = pd.DataFrame(comparisons)
    display(comparison[[
        "name", "winter", "base_days_30cm", "top_days_30cm",
        "same_grid_cell", "same_hourly_series",
    ]])
else:
    print("Pilot paused until the coordinates are filled in.")

,name,winter,base_days_30cm,top_days_30cm,same_grid_cell,same_hourly_series
0,Zermatt-Matterhorn,1995,136,136,False,False
1,Zermatt-Matterhorn,2025,136,136,False,False
2,St. Moritz - Corviglia,1995,136,136,True,True
3,St. Moritz - Corviglia,2025,131,131,True,True
4,Andermatt-Sedrun,1995,136,136,True,True
5,Andermatt-Sedrun,2025,136,136,True,True
6,Cervinia-Valtournenche,1995,136,136,False,False
7,Cervinia-Valtournenche,2025,136,136,False,False
8,Livigno,1995,136,136,True,True
9,Livigno,2025,129,129,True,True


## 4. Save the tables

Writing a CSV means saving the Python table as a file that Luca can open or import. The team run saves two files in `data/processed/`. The example saves its files in `data/raw/`, which the repository ignores.

In [5]:
if pilot_ready:
    out_dir = processed_dir if team_mode else raw_dir / "trafoi_example"
    out_dir.mkdir(parents=True, exist_ok=True)
    snow_winters.to_csv(out_dir / "snow_winters_pilot.csv", index=False)
    comparison.to_csv(out_dir / "snow_pilot_base_top.csv", index=False)
    identical = int(comparison["same_hourly_series"].sum())
    same_cell = int(comparison["same_grid_cell"].sum())
    total = len(comparison)
    print(f"Same grid cell: {same_cell}/{total}. Same hourly series: {identical}/{total}.")
    print(f"Saved both CSVs in {out_dir.relative_to(root)}.")

## Hand off the results

The pilot outputs are `snow_winters_pilot.csv` and `snow_pilot_base_top.csv` in `data/processed/`. The completed study dataset for these ten locations is `snow_winters_10resorts.csv`, and the period means are in `snow_summary_10resorts.csv`.

The base/top comparison reports the returned grid coordinates and whether the hourly series match. The source and location decisions are recorded in `docs/matthias_snow_run.md`.

**Data credit** · Open-Meteo Historical Weather API, using ECMWF/Copernicus ERA5-Land, CC BY 4.0. © OpenStreetMap contributors, ODbL. Measured station data source is MeteoSwiss.


## 5. Download the study winters

The next cell downloads the 20 study winters for the ten pilot locations. `RUN_DOWNLOAD_BATCH` is enabled for this completed run. Each uncached request is followed by a two-second pause, and responses are saved before the next step.

A complete run for the ten pilot resorts writes `data/processed/snow_winters_10resorts.csv`. If the group later supplies its full roster as `data/raw/resorts_full_input.csv`, the same cell uses that roster and writes `snow_winters.csv` once it is complete. Both files follow the team's `snow_winters` data dictionary.

Each batch is capped at 450 new season requests. For a large roster, run at most once per hour and twice in 24 hours, allowing for other requests your group makes. If the service rate-limits a batch, cached results remain available for the next run. [Open-Meteo call limits](https://open-meteo.com/en/terms)

In [6]:
RUN_DOWNLOAD_BATCH = True
MAX_NEW_REQUESTS = 450

if RUN_DOWNLOAD_BATCH:
    if not team_mode or not pilot_ready or "comparison" not in globals():
        raise ValueError("Run the team pilot with all 10 resorts before starting this batch.")

    full_csv = raw_dir / "resorts_full_input.csv"
    full_mode = full_csv.is_file()
    roster = read_resorts(full_csv, require_top=False) if full_mode else resorts
    unresolved = roster[coordinate_columns].isna().any(axis=1)
    if unresolved.any():
        ids = ", ".join(roster.loc[unresolved, "resort_id"].head(10))
        raise ValueError(f"Fill the missing base coordinates before downloading: {ids}")

    rows = []
    new_requests = 0
    rate_limited = False
    for resort in roster.itertuples(index=False):
        for winter in STUDY_WINTERS:
            _, cached_file, _, _ = snow_request(resort, winter, resort.elev_base_m)
            needs_download = not cached_file.is_file()
            if needs_download and (new_requests >= MAX_NEW_REQUESTS or rate_limited):
                continue
            if needs_download:
                new_requests += 1
                if new_requests % 50 == 0:
                    print(f"Starting new request {new_requests}.")
                sleep(2)
            try:
                snow, _ = get_hourly_snow(resort, winter, resort.elev_base_m)
            except HTTPError as error:
                if error.code != 429:
                    raise
                rate_limited = True
                continue
            rows.append(base_winter_row(resort, winter, count_snow_days(snow)))

    expected = len(roster) * len(STUDY_WINTERS)
    if rows:
        progress = pd.DataFrame(rows)
        progress.to_csv(raw_dir / "snow_winters_in_progress.csv", index=False)
        if len(rows) == expected:
            processed_dir.mkdir(parents=True, exist_ok=True)
            output_name = "snow_winters.csv" if full_mode else "snow_winters_10resorts.csv"
            progress.to_csv(processed_dir / output_name, index=False)
            print(f"Study snow dataset saved to data/processed/{output_name}.")
    print(f"Ready: {len(rows)}/{expected} resort winters. New requests: {new_requests}.")
    if rate_limited:
        print("Open-Meteo reached its rate limit. Run this cell again later.")
else:
    print("Base-only download is off. Enable it after reviewing the team pilot.")

## 6. Compare the two periods

This table averages ten winters per period for each pilot location. `lost_snow_days` equals the older average minus the recent average. Positive values indicate fewer snow days in the recent period.

The risk class uses the team's thresholds, applied to the recent mean. These ten selected pilot locations provide a working example for the final dataset.

In [7]:
study_file = processed_dir / "snow_winters_10resorts.csv"
if study_file.is_file():
    study = pd.read_csv(study_file)
    counts = study.groupby(["resort_id", "period"]).size()
    if (not counts.eq(10).all() or len(counts) != 2 * len(resorts)
                or set(study["resort_id"]) != set(resorts["resort_id"])):
        raise ValueError("Each pilot resort must have ten winters in each period.")
    means = study.groupby(["resort_id", "period"])["snow_days_30cm"].mean().unstack()
    summary = means.rename(columns={"past": "days_past", "recent": "days_recent"}).reset_index()
    summary["lost_snow_days"] = summary["days_past"] - summary["days_recent"]
    summary["risk_class"] = pd.cut(
        summary["days_recent"], bins=[-1, 60, 100, 200],
        labels=["critical", "at risk", "reliable"], right=False,
    )
    summary = resorts[["resort_id", "name", "country", "elev_base_m"]].merge(
        summary, on="resort_id", validate="one_to_one",
    )
    summary.to_csv(processed_dir / "snow_summary_10resorts.csv", index=False)
    display(summary)
else:
    print("The period comparison will appear when all 200 pilot resort winters are ready.")

,resort_id,name,country,elev_base_m,days_past,days_recent,lost_snow_days,risk_class
0,zermatt-matterhorn,Zermatt-Matterhorn,CH,1562,136.2,136.3,-0.1,reliable
1,st-moritz-corviglia,St. Moritz - Corviglia,CH,1772,136.1,132.6,3.5,reliable
2,andermatt-sedrun-disentis,Andermatt-Sedrun,CH,1444,136.2,136.3,-0.1,reliable
3,breuil-cervinia-valtournenche-zermatt,Cervinia-Valtournenche,IT,1562,131.6,132.2,-0.6,reliable
4,livigno,Livigno,IT,1816,135.4,132.9,2.5,reliable
5,aprica,Aprica,IT,1162,123.2,113.5,9.7,reliable
6,kitzbuehel-kirchberg,Kitzbuehel,AT,800,119.0,124.5,-5.5,reliable
7,soelden,Soelden,AT,1350,136.0,135.8,0.2,reliable
8,les-houches-saint-gervais-chamonix,Chamonix,FR,1000,124.5,135.2,-10.7,reliable
9,les-2-alpes,Les 2 Alpes,FR,1280,128.4,132.1,-3.7,reliable


## Swiss station data · Ready to use

The companion `meteoswiss_station_winters.csv` contains daily snow-depth counts for the MeteoSwiss manual stations above 1,000 m with complete observations in all 20 study winters. The coverage check retained **20 stations** and **400 station winters**. Put that CSV in `data/processed/`. To rebuild it from the source files, set `RUN_STATION_DOWNLOAD = True` below and run that cell after setup.

MeteoSwiss records `hto000d0` once daily at **06:00 UTC**, in centimeters. The file counts observations at or above 30 cm. The Open-Meteo resort counts above use a daily maximum. For SQ2, compare the station reading against the **06:00 UTC** Open-Meteo value at the same station coordinates and on the same date. This keeps measurement time consistent.

The CSV is a complete-record sample for comparison. Check the station histories before using the series to infer long-run changes. [Source: MeteoSwiss manual precipitation stations](https://opendatadocs.meteoswiss.ch/a-data-groundbased/a5-manual-precipitation-stations) · [Snow measurement FAQ](https://opendatadocs.meteoswiss.ch/general/faq#snow-measurement-data--what-data-is-available-and-where-can-i-find-it)

In [8]:
RUN_STATION_DOWNLOAD = False

if RUN_STATION_DOWNLOAD:
    meteo_base = "https://data.geo.admin.ch/ch.meteoschweiz.ogd-nime"
    meteo_cache = raw_dir / "meteoswiss"
    meteo_cache.mkdir(parents=True, exist_ok=True)


    def read_meteoswiss(relative_path):
        cache_file = meteo_cache / Path(relative_path).name
        if not cache_file.is_file():
            with urlopen(f"{meteo_base}/{relative_path}", timeout=45) as response:
                data = response.read()
            temp_file = cache_file.with_suffix(".tmp")
            temp_file.write_bytes(data)
            temp_file.replace(cache_file)
            sleep(0.3)
        return pd.read_csv(cache_file, sep=";", encoding="latin1")


    snow_parameter = "hto000d0"
    parameters = read_meteoswiss("ogd-nime_meta_parameters.csv").set_index("parameter_shortname")
    if parameters.loc[snow_parameter, "parameter_unit"] != "cm":
        raise ValueError("Check the unit of the MeteoSwiss snow-depth parameter before counting.")

    stations = read_meteoswiss("ogd-nime_meta_stations.csv").rename(columns={
        "station_abbr": "station_id",
        "station_height_masl": "elev_m",
        "station_coordinates_wgs84_lat": "lat",
        "station_coordinates_wgs84_lon": "lon",
    })
    inventory = read_meteoswiss("ogd-nime_meta_datainventory.csv")
    inventory = inventory.loc[inventory["parameter_shortname"].eq(snow_parameter)].copy()
    date_format = "%d.%m.%Y %H:%M"
    inventory["starts"] = pd.to_datetime(inventory["data_since"], format=date_format)
    inventory["ends"] = pd.to_datetime(inventory["data_till"], format=date_format).fillna(pd.Timestamp.today())
    coverage = inventory.groupby("station_abbr").agg(starts=("starts", "min"), ends=("ends", "max"))
    first_day = pd.Timestamp(f"{STUDY_WINTERS[0] - 1}-12-01")
    last_day = pd.Timestamp(f"{STUDY_WINTERS[-1]}-04-15")
    long_records = coverage.index[coverage["starts"].le(first_day) & coverage["ends"].ge(last_day)]
    candidates = stations.loc[
        stations["station_id"].isin(long_records) & stations["elev_m"].gt(1000),
        ["station_id", "station_name", "lat", "lon", "elev_m"],
    ].dropna().sort_values("station_id")
    winter_dates = {
        year: pd.date_range(f"{year - 1}-12-01", f"{year}-04-15") for year in STUDY_WINTERS
    }
    study_dates = pd.DatetimeIndex([]).append(list(winter_dates.values()))

    station_rows = []
    excluded = []
    for station in candidates.itertuples(index=False):
        code = station.station_id.lower()
        try:
            observations = read_meteoswiss(f"{code}/ogd-nime_{code}_d_historical.csv")
        except HTTPError as error:
            if error.code != 404:
                raise
            excluded.append((station.station_id, "historical daily file missing"))
            continue

        dates = pd.to_datetime(observations["reference_timestamp"], format=date_format).dt.normalize()
        depth_cm = pd.to_numeric(observations[snow_parameter], errors="coerce")
        depth_by_day = pd.Series(depth_cm.where(depth_cm.ge(0)).to_numpy(), index=dates)
        depth_by_day = depth_by_day.loc[depth_by_day.index.isin(study_dates)]
        if depth_by_day.index.has_duplicates:
            excluded.append((station.station_id, "duplicate dates in study windows"))
            continue
        winters = {year: depth_by_day.reindex(days) for year, days in winter_dates.items()}
        complete = sum(not values.isna().any() for values in winters.values())
        if complete != len(STUDY_WINTERS):
            excluded.append((station.station_id, f"{complete} complete winters"))
            continue

        for winter, values in winters.items():
            station_rows.append({
                "station_id": station.station_id,
                "lat": station.lat,
                "lon": station.lon,
                "elev_m": station.elev_m,
                "winter": winter,
                "snow_days_30cm": int(values.ge(30).sum()),
                "station_name": station.station_name,
                "period": "past" if winter <= 1995 else "recent",
                "observed_days": len(values),
                "source": "MeteoSwiss",
            })

    if not station_rows:
        raise ValueError("No MeteoSwiss stations had complete data across all 20 study winters.")
    station_winters = pd.DataFrame(station_rows)
    processed_dir.mkdir(parents=True, exist_ok=True)
    station_winters.to_csv(processed_dir / "meteoswiss_station_winters.csv", index=False)
    print(f"{len(candidates)} candidate stations above 1,000 m. "
          f"{station_winters['station_id'].nunique()} complete stations, {len(station_winters)} station winters.")
    if excluded:
        display(pd.DataFrame(excluded, columns=["station_id", "reason"]))
    display(station_winters.head())
else:
    station_file = processed_dir / "meteoswiss_station_winters.csv"
    if station_file.is_file():
        station_winters = pd.read_csv(station_file)
        print(f"Measured station data ready: {len(station_winters)} rows, "
              f"{station_winters['station_id'].nunique()} stations.")
        display(station_winters.head())
    else:
        print("Enable RUN_STATION_DOWNLOAD to build the measured-station CSV.")

,station_id,lat,lon,elev_m,winter,snow_days_30cm,station_name,period,observed_days,source
0,ABO,46.491703,7.560703,1321.0,1986,63,Adelboden,past,136,MeteoSwiss
1,ABO,46.491703,7.560703,1321.0,1987,112,Adelboden,past,136,MeteoSwiss
2,ABO,46.491703,7.560703,1321.0,1988,57,Adelboden,past,137,MeteoSwiss
3,ABO,46.491703,7.560703,1321.0,1989,17,Adelboden,past,136,MeteoSwiss
4,ABO,46.491703,7.560703,1321.0,1990,2,Adelboden,past,136,MeteoSwiss
